In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

import scripts.feature_engineering as feature
from scripts.evaluate import CLIP, calculate_metrics
from scripts.prepare_data import append_test_month

In [2]:
df = pd.read_csv('data/gold/train.csv')
df_cat = pd.read_csv('data/landing/item_categories.csv')
df_items = pd.read_csv('data/landing/items.csv')
df_shops = pd.read_csv('data/landing/shops.csv')
df_sales = pd.read_csv('data/landing/sales_train.csv')
sample = pd.read_csv('data/landing/sample_submission.csv')

# тест как месяц 34: target = NaN, ID сохраняется; магазины-дубли переобозначаются как в train
df = append_test_month(df, 'data/landing/test.csv')

df = feature.add_categories(df, df_cat, df_items)
df = feature.add_cities(df, df_shops)
df = feature.add_target_lags(df, (1, 2, 3, 4, 5, 6, 12))
df = feature.add_lag_stats(df)
df = feature.add_mean_lags(df)
df = feature.add_age_features(df)
df = feature.add_price_features(df, df_sales, df_items)
df = feature.add_calendar(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11098754 entries, 0 to 11098753
Data columns (total 35 columns):
 #   Column                Dtype  
---  ------                -----  
 0   date_block_num        int64  
 1   shop_id               int64  
 2   item_id               int64  
 3   target                float64
 4   ID                    float64
 5   item_category_id      uint8  
 6   category_type_id      uint8  
 7   item_type_id          uint8  
 8   city_id               uint8  
 9   target_lag_1          float32
 10  target_lag_2          float32
 11  target_lag_3          float32
 12  target_lag_4          float32
 13  target_lag_5          float32
 14  target_lag_6          float32
 15  target_lag_12         float32
 16  roll3                 float32
 17  roll6                 float32
 18  diff_1_2              float32
 19  item_mean_lag_1       float32
 20  shop_mean_lag_1       float32
 21  cat_mean_lag_1        float32
 22  shop_cat_mean_lag_1   float32
 23  city_item_mean_l

In [3]:
CAT_FEATURES = ['shop_id', 'item_category_id', 'category_type_id', 'item_type_id', 'city_id']
FEATURES = [c for c in df.columns if c not in ('target', 'ID', 'date_block_num', 'item_id')]

m = df['date_block_num']
test = df[m == 34]
assert len(test) == len(sample), f'в месяце 34 {len(test)} строк, в sample_submission {len(sample)}'
assert test['ID'].notna().all() and df.loc[m < 34, 'ID'].isna().all()

check = pd.DataFrame({
    'NaN, месяц 33': df.loc[m == 33, FEATURES].isna().mean(),
    'NaN, месяц 34': test[FEATURES].isna().mean(),
    'среднее, месяц 33': df.loc[m == 33, FEATURES].mean(),
    'среднее, месяц 34': test[FEATURES].mean(),
})
print('новых товаров в тесте:', int(test['item_is_new'].sum()), 'пар из', len(test))
check.round(3)

новых товаров в тесте: 15246 пар из 214200


,"NaN, месяц 33","NaN, месяц 34","среднее, месяц 33","среднее, месяц 34"
shop_id,0.000,0.000,30.864,31.643
item_category_id,0.000,0.000,46.992,46.310
category_type_id,0.000,0.000,12.171,12.043
item_type_id,0.000,0.000,24.607,24.292
city_id,0.000,0.000,15.068,15.095
target_lag_1,0.304,0.217,0.327,0.327
target_lag_2,0.334,0.305,0.365,0.354
target_lag_3,0.360,0.339,0.338,0.393
target_lag_4,0.396,0.366,0.350,0.359
target_lag_5,0.417,0.399,0.349,0.369


In [4]:
PARAMS = dict(
    objective='regression', metric='rmse',
    n_estimators=215, learning_rate=0.05,
    num_leaves=84, min_child_samples=236,
    colsample_bytree=0.6645097995051339, subsample=0.5756895436932168, subsample_freq=1,
    reg_lambda=2.8570139411689, cat_smooth=5.178938561126461,
    random_state=42, n_jobs=-1, verbose=-1,
)


def fit(data: pd.DataFrame) -> lgb.LGBMRegressor:
    model = lgb.LGBMRegressor(**PARAMS)
    return model.fit(data[FEATURES], data['target'], categorical_feature=CAT_FEATURES)

In [5]:
train33, test33 = df[(m >= 12) & (m <= 32)], df[m == 33]
model33 = fit(train33)

res33 = calculate_metrics(
    model33,
    {'test': (test33[FEATURES], test33['target'])},
    model_name='LightGBM, месяц 33',
    baselines={'zeros': np.zeros(len(test33)), 'naive': test33['target_lag_1'].fillna(0)},
    baseline_split='test',
)
res33.round(4)

,rmse,mae,bias,r2,rmse_nonzero
test,0.8747,0.3052,0.0148,0.4075,2.1574
test: zeros,1.1654,0.2587,-0.2587,-0.0518,3.2055
test: naive,1.1360,0.3020,-0.0312,0.0006,2.8703


In [6]:
final = fit(df[(m >= 12) & (m <= 33)])

submission = pd.DataFrame({
    'ID': test['ID'].astype('int64').to_numpy(),
    'item_cnt_month': np.clip(final.predict(test[FEATURES]), *CLIP),
})
# порядок строк как в sample_submission
submission = sample[['ID']].merge(submission, on='ID', how='left', validate='one_to_one')
submission['item_cnt_month'].describe()

count    214200.000000
mean          0.295476
std           0.860882
min           0.000000
25%           0.020743
50%           0.078770
75%           0.250789
max          20.000000
Name: item_cnt_month, dtype: float64

In [7]:
assert list(submission.columns) == list(sample.columns), 'столбцы не совпадают'
assert len(submission) == len(sample), 'число строк не совпадает'
assert submission['ID'].equals(sample['ID']), 'ID или их порядок не совпадают'
assert submission['ID'].dtype == sample['ID'].dtype, 'тип ID не совпадает'
assert submission['item_cnt_month'].notna().all(), 'есть пропуски в прогнозе'
assert submission['item_cnt_month'].between(*CLIP).all(), 'прогноз вне [0, 20]'

out = Path('data/output')
out.mkdir(parents=True, exist_ok=True)
path = out / 'submission_lgbm.csv'
submission.to_csv(path, index=False)

saved = pd.read_csv(path)
assert list(saved.columns) == list(sample.columns) and saved['ID'].equals(sample['ID'])
print(f'сохранено: {path}, {len(saved)} строк')
saved.head()

сохранено: data\output\submission_lgbm.csv, 214200 строк


,ID,item_cnt_month
0,0,0.538653
1,1,0.436266
2,2,1.105090
3,3,0.473665
4,4,3.202214
